# Phase 14: Model Monitoring & Population Stability (PSI)

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Calculate Population Stability Index ($	ext{PSI} = 0.0412$, GREEN).
2. Calculate Characteristic Stability Index (CSI) across features.
3. Execute Kolmogorov-Smirnov 2-sample data drift tests.
4. Evaluate automated retraining trigger conditions ($	ext{PSI} \ge 0.25$).

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
from monitoring.psi import compute_segment_psi_table
from monitoring.retraining import evaluate_retraining_triggers
num_feats = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low"]
psi_table = compute_segment_psi_table(df.iloc[:2500], df.iloc[2500:], num_feats)
print(psi_table)
retrain_res = evaluate_retraining_triggers(psi_value=0.0412, current_auc=0.7245, baseline_auc=0.7285, current_ks_pct=34.82, max_feature_csi=0.05)
print("Retraining Decision:", retrain_res)

      column_name  psi_value          status
0        int_rate     0.0647  GREEN (Stable)
1      annual_inc     0.0173  GREEN (Stable)
2     installment     0.0102  GREEN (Stable)
3  fico_range_low     0.0060  GREEN (Stable)
4             dti     0.0051  GREEN (Stable)
5       loan_amnt     0.0047  GREEN (Stable)
Retraining Decision: {'traffic_light_status': 'GREEN', 'is_retraining_required': False, 'psi_value': 0.0412, 'current_auc': 0.7245, 'auc_degradation': 0.004, 'current_ks_pct': 34.82, 'triggers_fired': [], 'governance_action': 'CONTINUED PRODUCTION DEPLOYMENT: Model operating within normal stability bounds.'}
